# RWI vs pobreza IDB — BRA + MEX + ARG (Colab notebook)

**Por qué este notebook existe**: los rasters WorldPop 100m de BRA (10 GB), MEX (3.3 GB) y ARG (3.8 GB) no caben en memoria de una máquina local típica. Este notebook implementa lectura **ventaneada** (windowed reading): en lugar de cargar el raster entero a RAM, lee solo la ventana de pixeles alrededor de cada celda RWI (~27×27 pixeles = ~3 KB por celda).

**Inputs requeridos** (deben estar accesibles desde Colab — sea por Drive mount o subida directa):

| Archivo | Tamaño aprox | Path esperado |
|---|---|---|
| `bra_relative_wealth_index.csv` (RWI Meta) | 9 MB | `data/Poverty Rates/meta-rwi/BRA/` |
| `bra_pop_2023_CN_100m_R2025A_v1.tif` (WorldPop) | 480 MB | `data/population/WorldPop/BRA/` |
| `mex_relative_wealth_index.csv` | 25 MB | `data/Poverty Rates/meta-rwi/MEX/` |
| `mex_pop_2023_CN_100m_R2025A_v1.tif` | 150 MB | `data/population/WorldPop/MEX/` |
| `arg_relative_wealth_index.csv` | 9 MB | `data/Poverty Rates/meta-rwi/ARG/` |
| `arg_pop_2023_CN_100m_R2025A_v1.tif` | 300 MB | `data/population/WorldPop/ARG/` |
| `lac-level-2.shp` (+ `.dbf .prj .shx .cpg`) | 17 MB | `data/bounderys/LAC/level 2/` |
| `lac-level-2.csv` (Pobreza BID) | 1 MB | `data/Poverty Rates/` |

**Total**: ~1 GB. Cabe en Colab Free tier sin problemas.

**Outputs** (al terminar, descargar y colocar en `results/exploratory/rwi_vs_poverty/` del repo local):
- `{ISO}_rwi_adm2_aggregated.csv`
- `{ISO}_rwi_vs_poverty_merged.csv`
- `{ISO}_rwi_vs_poverty_tests.csv` ← este es el que lee el dashboard

Luego correr `uv run python pipeline/export_dashboard_data.py` para refrescar el payload.


## 1. Setup

Hay dos opciones para acceder a los datos:

**Opción A — Google Drive (recomendada)**: tener la carpeta del proyecto sincronizada con Drive. La celda siguiente la monta. Asume que el proyecto vive en `MyDrive/IDB/accessibility_platform/`. Cambiar `PROJECT_ROOT` si la ruta es distinta.

**Opción B — Subida directa**: arrastrar los archivos al panel "Files" de Colab manteniendo la estructura `data/Poverty Rates/meta-rwi/{ISO}/...` y `data/population/WorldPop/{ISO}/...`. Saltarse la celda de Drive y poner `PROJECT_ROOT = "/content"`.

In [ ]:
# --- Install dependencies (Colab) ---
!pip install -q rasterio geopandas scipy

In [ ]:
# --- Opción A: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/IDB/accessibility_platform'
# Si tu carpeta vive en otra ruta de Drive, ajustá esta línea.

import os
assert os.path.isdir(PROJECT_ROOT), f'No existe {PROJECT_ROOT} — revisar la ruta'
print(f'PROJECT_ROOT OK: {PROJECT_ROOT}')

In [ ]:
# --- Paths fijos derivados del PROJECT_ROOT ---
from pathlib import Path

ROOT = Path(PROJECT_ROOT)
RWI_DIR = ROOT / 'data' / 'Poverty Rates' / 'meta-rwi'
POP_DIR = ROOT / 'data' / 'population' / 'WorldPop'
ADM2_SHP = ROOT / 'data' / 'bounderys' / 'LAC' / 'level 2' / 'lac-level-2.shp'
POVERTY_CSV = ROOT / 'data' / 'Poverty Rates' / 'lac-level-2.csv'
OUT_DIR = ROOT / 'results' / 'exploratory' / 'rwi_vs_poverty'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [RWI_DIR, POP_DIR, ADM2_SHP, POVERTY_CSV]:
    assert p.exists(), f'No existe: {p}'
print('Todos los paths OK')

# RWI Bing quadkey level-14 tile size (degrees) — invariante del dataset Meta
RWI_TILE_DEG = 0.02148

## 2. Función ventaneada de agregación

**Por qué ventaneada**: el script local `pipeline/06_pop_exploratory.py` hace `src.read(1)` que carga el raster entero (10 GB para BRA → MemoryError). Esta versión usa `rasterio.windows.Window` para leer solo los ~3 KB que cubren cada celda RWI individual.

Tiempo esperado: ~2-5 min por país (vs segundos en el caso pequeño, pero 100× menos memoria).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from scipy.stats import spearmanr, pearsonr


def aggregate_rwi_by_adm2_windowed(iso: str) -> pd.DataFrame:
    """Same as pipeline/06_pop_exploratory.py::aggregate_rwi_by_adm2 but
    reads the WorldPop raster in windows around each RWI cell rather than
    loading the full array. Allows BRA / MEX / ARG / other large countries
    to run on machines with limited RAM.
    """
    rwi_path = RWI_DIR / iso / f'{iso.lower()}_relative_wealth_index.csv'
    pop_path = POP_DIR / iso / f'{iso.lower()}_pop_2023_CN_100m_R2025A_v1.tif'
    print(f'[{iso}] RWI:      {rwi_path.name}')
    print(f'[{iso}] WorldPop: {pop_path.name}')

    rwi = pd.read_csv(rwi_path)
    print(f'[{iso}] {len(rwi):,} RWI cells loaded')

    with rasterio.open(pop_path) as src:
        if str(src.crs).upper() != 'EPSG:4326':
            raise ValueError(f'WorldPop raster must be EPSG:4326, got {src.crs}')
        transform = src.transform
        nodata = src.nodata if src.nodata is not None else -99999
        width, height = src.width, src.height
        pixel_w = transform.a
        pixel_h = -transform.e
        half = RWI_TILE_DEG / 2.0
        half_px_x = max(1, int(round(half / pixel_w)))
        half_px_y = max(1, int(round(half / pixel_h)))
        print(f'[{iso}] Raster {width}x{height} (would need {width*height*4/1e9:.2f} GB if loaded full)')
        print(f'[{iso}] Window per RWI cell: {2*half_px_x+1} x {2*half_px_y+1} pixels')

        lons = rwi['longitude'].to_numpy()
        lats = rwi['latitude'].to_numpy()
        cx = np.round((lons - transform.c) / pixel_w).astype(np.int64)
        cy = np.round((transform.f - lats) / pixel_h).astype(np.int64)

        pop_per_cell = np.zeros(len(rwi), dtype=np.float64)
        for i in range(len(rwi)):
            c0 = max(0, cx[i] - half_px_x)
            c1 = min(width, cx[i] + half_px_x + 1)
            r0 = max(0, cy[i] - half_px_y)
            r1 = min(height, cy[i] + half_px_y + 1)
            if c1 > c0 and r1 > r0:
                w = Window(c0, r0, c1 - c0, r1 - r0)
                arr = src.read(1, window=w)
                arr = np.where(arr == nodata, 0.0, arr)
                arr = np.where(arr < 0, 0.0, arr)
                pop_per_cell[i] = float(arr.sum())
            if (i + 1) % 10000 == 0:
                print(f'[{iso}]   processed {i+1:,} / {len(rwi):,} cells')

    rwi['pop'] = pop_per_cell
    print(f'[{iso}] Population captured by RWI cells: {pop_per_cell.sum():,.0f}')
    print(f'[{iso}] RWI cells with pop > 0: {(pop_per_cell > 0).sum():,} / {len(rwi):,}')

    adm2 = gpd.read_file(ADM2_SHP)
    adm2 = adm2[adm2['ADM0_PCODE'] == iso].copy()
    print(f'[{iso}] {len(adm2):,} ADM2 polygons')

    rwi_gdf = gpd.GeoDataFrame(
        rwi,
        geometry=gpd.points_from_xy(rwi['longitude'], rwi['latitude']),
        crs='EPSG:4326',
    )
    rwi_joined = gpd.sjoin(rwi_gdf, adm2[['ADM2_PCODE', 'geometry']],
                           how='left', predicate='within')
    n_unmatched = rwi_joined['ADM2_PCODE'].isna().sum()
    print(f'[{iso}] RWI cells outside any ADM2: {n_unmatched:,} ({n_unmatched/len(rwi):.1%})')
    rwi_joined = rwi_joined.dropna(subset=['ADM2_PCODE'])

    def weighted_stats(g):
        w = g['pop'].to_numpy()
        x = g['rwi'].to_numpy()
        e = g['error'].to_numpy()
        sw = float(w.sum())
        if sw <= 0:
            return pd.Series({
                'n_rwi_cells': len(g),
                'pop_total': 0.0,
                'rwi_pop_weighted_mean': np.nan,
                'rwi_pop_weighted_var': np.nan,
                'rwi_unweighted_mean': float(x.mean()) if len(x) else np.nan,
                'error_mean': float(e.mean()) if len(e) else np.nan,
            })
        mean = float((w * x).sum() / sw)
        var = float((w * (x - mean) ** 2).sum() / sw)
        return pd.Series({
            'n_rwi_cells': len(g),
            'pop_total': sw,
            'rwi_pop_weighted_mean': mean,
            'rwi_pop_weighted_var': var,
            'rwi_unweighted_mean': float(x.mean()),
            'error_mean': float(e.mean()),
        })

    agg = rwi_joined.groupby('ADM2_PCODE').apply(weighted_stats).reset_index()

    adm2_min = adm2[['ADM0_PCODE','ADM1_PCODE','ADM1_EN','ADM2_PCODE','ADM2_EN']].copy()
    agg = adm2_min.merge(agg, on='ADM2_PCODE', how='left')
    agg['n_rwi_cells'] = agg['n_rwi_cells'].fillna(0).astype(int)
    agg['pop_total'] = agg['pop_total'].fillna(0.0)
    print(f'[{iso}] ADM2 covered: {(agg["n_rwi_cells"] > 0).sum()}')

    out_path = OUT_DIR / f'{iso}_rwi_adm2_aggregated.csv'
    agg.to_csv(out_path, index=False)
    print(f'[{iso}] Wrote {out_path}')
    return agg

In [ ]:
def run_correlation_tests(iso: str, n_boot: int = 1000, seed: int = 42) -> pd.DataFrame:
    """Replica de pipeline/06_pop_exploratory.py::run_correlation_tests, robusta
    a países sin POVERTY_RATE o sin NBI_RATE (skip cuando n<2 en lugar de crash)."""
    agg_path = OUT_DIR / f'{iso}_rwi_adm2_aggregated.csv'
    assert agg_path.exists(), f'Run aggregate first; missing {agg_path}'
    agg = pd.read_csv(agg_path)
    poverty = pd.read_csv(POVERTY_CSV)
    poverty = poverty[poverty['ADM0_PCODE'] == iso].copy()

    df = agg.merge(
        poverty[['ADM2_PCODE', 'POVERTY_RATE', 'NBI_RATE', 'POVERTY_NUM', 'NBI_NUM']],
        on='ADM2_PCODE',
        how='inner',
    )
    df = df[(df['n_rwi_cells'] >= 5) & (df['pop_total'] > 0)].copy()
    print(f'[{iso}] Test sample after filter: {len(df):,} ADM2')

    rng = np.random.default_rng(seed)
    rows = []
    for target in ('POVERTY_RATE', 'NBI_RATE'):
        sub = df[['rwi_pop_weighted_mean', target]].dropna().copy()
        if len(sub) < 2:
            print(f'[{iso}] {target}: n={len(sub)} < 2 — skipping')
            continue
        x = sub['rwi_pop_weighted_mean'].to_numpy()
        y = sub[target].to_numpy()
        rho_s, p_s = spearmanr(x, y)
        rho_p, p_p = pearsonr(x, y)
        boots = np.empty(n_boot)
        idx = np.arange(len(sub))
        for b in range(n_boot):
            sample = rng.choice(idx, size=len(idx), replace=True)
            r, _ = spearmanr(x[sample], y[sample])
            boots[b] = r
        ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])

        sub['rwi_decile'] = pd.qcut(sub['rwi_pop_weighted_mean'], 10, labels=False, duplicates='drop')
        sub['target_decile'] = pd.qcut(sub[target], 10, labels=False, duplicates='drop')
        sub['rwi_decile_inv'] = 9 - sub['rwi_decile']
        agreement = (sub['rwi_decile_inv'] == sub['target_decile']).mean()
        within_one = (np.abs(sub['rwi_decile_inv'] - sub['target_decile']) <= 1).mean()

        rows.append({
            'country': iso,
            'target': target,
            'n': len(sub),
            'spearman_rho': float(rho_s),
            'spearman_p': float(p_s),
            'spearman_ci_lo': float(ci_lo),
            'spearman_ci_hi': float(ci_hi),
            'pearson_r': float(rho_p),
            'pearson_p': float(p_p),
            'decile_exact_match': float(agreement),
            'decile_within_one': float(within_one),
        })
        print(f'[{iso}] vs {target}: Spearman rho = {rho_s:.3f} [{ci_lo:.3f}, {ci_hi:.3f}], n={len(sub)}, decile match = {agreement:.1%}')

    tests_df = pd.DataFrame(rows)
    tests_path = OUT_DIR / f'{iso}_rwi_vs_poverty_tests.csv'
    tests_df.to_csv(tests_path, index=False)
    print(f'[{iso}] Wrote {tests_path}')

    merged = df.copy()
    merged_path = OUT_DIR / f'{iso}_rwi_vs_poverty_merged.csv'
    merged.to_csv(merged_path, index=False)
    print(f'[{iso}] Wrote {merged_path}')
    return tests_df

## 3. Correr BRA, MEX, ARG

Cada país tarda ~5-15 minutos en Colab CPU. BRA es el más pesado por número de celdas RWI (173K).

In [ ]:
import time

for iso in ['ARG', 'MEX', 'BRA']:  # de menor a mayor; BRA al final
    print(f'\n=== {iso} ===')
    t0 = time.time()
    agg = aggregate_rwi_by_adm2_windowed(iso)
    print(f'[{iso}] aggregate done in {time.time()-t0:.1f}s')
    t0 = time.time()
    tests = run_correlation_tests(iso)
    print(f'[{iso}] tests done in {time.time()-t0:.1f}s')

## 4. Verificar outputs

Después de la corrida, los archivos `{ISO}_rwi_vs_poverty_tests.csv` deben aparecer en `results/exploratory/rwi_vs_poverty/` (sea en Drive si montaste Drive, sea en `/content/` si subiste). Si estuviste con Drive mount, los archivos quedan automáticamente sincronizados en tu computadora local.

**Si subiste manualmente (Opción B)**, ejecutá la celda siguiente para descargar los 3 tests.csv.

In [ ]:
# Solo si NO usaste Drive mount
from google.colab import files as colab_files
for iso in ['ARG', 'MEX', 'BRA']:
    p = OUT_DIR / f'{iso}_rwi_vs_poverty_tests.csv'
    if p.exists():
        colab_files.download(str(p))
        # también los merged para si querés re-correr análisis posteriores
        p2 = OUT_DIR / f'{iso}_rwi_vs_poverty_merged.csv'
        if p2.exists(): colab_files.download(str(p2))
    else:
        print(f'No existe {p}')

## 5. Paso siguiente (local)

Una vez que tenés los 3 archivos `tests.csv` (BRA, MEX, ARG) en `results/exploratory/rwi_vs_poverty/` de tu repo local:

```bash
uv run python pipeline/export_dashboard_data.py
cp results/dashboard/dashboard_payload.json ../accessibility-dashboard/content/dashboard-payload.json
```

El dashboard `step-06 → RWI multi-país` automáticamente reclasifica BRA + MEX + ARG de "pendiente"/"bloqueado" a "validado" con sus ρ Spearman.
